# 20 — Speed: jev-lite vs NLI vs Laya (local, Apple Silicon) vs JEV (API)

How fast each model answers the same records on this Mac. Local models run on MPS; JEV is timed over its API.

- **Latency:** one record per call, one call at a time — the time a judge waits for one answer. Includes tokenisation and the read-back of probabilities.
- **Throughput:** the same records grouped the way `scripts/predict_baseline.py` runs them (questions on the same state in one call; NLI: 64 states per batch).
- **JEV:** by default, the latencies recorded during the API run (`data/jev_api/preds.jsonl`, 8 requests in flight). That is a network round trip plus server time. `RUN_JEV_LIVE = True` times fresh calls one at a time instead (**costs API calls**).

Records: a stratified subset of the JEV API sample, the same for every model. Results go to `data/speed/apple_silicon/`.

Before running: plug in the power, close heavy apps (thermal throttling and memory pressure change the numbers).

## 1. Environment

In [1]:
STACK = "apple_silicon"

import os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")  # all three local models are in the Hugging Face cache

from jevlite.config import apply_env, load_config
from jevlite.log import logger, setup_logging

setup_logging()
cfg = load_config(STACK)
apply_env(cfg)
logger.info("stack {}: config loaded, root {}", STACK, ROOT)

10:47:49 | INFO    | stack apple_silicon: config loaded, root /Users/nefelim584/PycharmProjects/JEV_Judge


In [2]:
import platform
import subprocess

import torch


def _sysctl(key: str) -> str:
    return subprocess.run(["sysctl", "-n", key], capture_output=True, text=True).stdout.strip()


ENV = {
    "chip": _sysctl("machdep.cpu.brand_string"),
    "memory_gb": round(int(_sysctl("hw.memsize") or 0) / 2**30),
    "macos": platform.mac_ver()[0],
    "python": platform.python_version(),
    "torch": torch.__version__,
    "mps": torch.backends.mps.is_available(),
}
assert ENV["mps"], "MPS is not available: this notebook is for Apple Silicon"
ENV

{'chip': 'Apple M1 Pro',
 'memory_gb': 16,
 'macos': '26.5.2',
 'python': '3.14.0',
 'torch': '2.14.1',
 'mps': True}

## 2. Parameters

In [3]:
SAMPLE = "data/jev_api/sample.jsonl"
JEV_PREDS = "data/jev_api/preds.jsonl"
JEVLITE_MODEL = "data/runs/stage_a-full-s42/final"

LOCAL_MODELS = ["jevlite", "nli", "laya"]
N_PER_PRIMITIVE = 100  # records per Bool / Choice / Score, the same for every model
N_WARMUP = 10          # untimed calls first: MPS compiles kernels on the first shapes it sees
SEED = 0

RUN_JEV_LIVE = False   # True: time N_JEV_LIVE fresh API calls, one at a time (costs API calls)
N_JEV_LIVE = 60
JEV_MODEL = "jev-latest"

OUT = Path("data/speed/apple_silicon")
OUT.mkdir(parents=True, exist_ok=True)

## 3. Records

In [4]:
import random

import pandas as pd

from jevlite.data.unified import load_records
from jevlite.encoding import serialize_state

records = list(load_records(SAMPLE))
rng = random.Random(SEED)
bench = []
for prim in ("bool", "choice", "score"):
    pool = sorted((r for r in records if r.type == prim), key=lambda r: r.id)
    bench += rng.sample(pool, min(N_PER_PRIMITIVE, len(pool)))


def n_chars(r) -> int:
    return len(serialize_state(r.state)) + len(r.prompt) + sum(len(c) for c in r.candidates or [])


meta = pd.DataFrame([{"id": r.id, "primitive": r.type, "source": r.source, "chars": n_chars(r),
                      "options": len(r.candidates or [])} for r in bench])
meta.groupby("primitive").agg(n=("id", "size"), chars_p50=("chars", "median"), chars_p90=("chars", lambda s: s.quantile(0.9)),
                              options_max=("options", "max"))

,n,chars_p50,chars_p90,options_max
primitive,,,,
bool,100,576.5,1952.6,0
choice,100,290.0,1225.6,16
score,100,1268.5,1985.0,7


## 4. Local models

One model at a time, freed before the next. `memory_gb` is the MPS driver allocation after loading and after the run.

In [5]:
import gc
import time

from jevlite.baselines import load_baseline
from jevlite.baselines.base import iter_predictions


def free_mps():
    gc.collect()
    torch.mps.empty_cache()


def bench_local(name: str) -> tuple[list[dict], dict]:
    free_mps()
    t0 = time.perf_counter()
    predictor = load_baseline(name, JEVLITE_MODEL if name == "jevlite" else None, "mps")
    load_s = time.perf_counter() - t0
    mem_loaded = torch.mps.driver_allocated_memory() / 2**30

    for r in bench[:N_WARMUP]:
        try:
            predictor.predict(r.state, [r.to_question()])
        except Exception:
            pass

    rows = []
    for r in bench:
        q = r.to_question()
        t = time.perf_counter()
        try:
            predictor.predict(r.state, [q])
            error = None
        except Exception as e:  # e.g. Laya: options longer than its header
            error = f"{type(e).__name__}: {e}"
        rows.append({"model": name, "id": r.id, "latency_ms": (time.perf_counter() - t) * 1000, "error": error})

    t = time.perf_counter()
    done = sum(len(p) for p, _ in iter_predictions(predictor, bench, skip_errors=True, block=64))
    throughput = done / (time.perf_counter() - t)
    info = {"model": name, "predictor": predictor.name, "load_s": load_s, "memory_gb_loaded": mem_loaded,
            "memory_gb_after": torch.mps.driver_allocated_memory() / 2**30, "throughput_rec_s": throughput,
            "errors": sum(r["error"] is not None for r in rows)}
    del predictor
    free_mps()
    logger.info("{}: load {:.1f}s, {:.1f} rec/s grouped, {} errors", name, load_s, throughput, info["errors"])
    return rows, info


latency_rows, infos = [], []
for name in LOCAL_MODELS:
    rows, info = bench_local(name)
    latency_rows += rows
    infos.append(info)
pd.DataFrame(infos).set_index("model").round(2)

/Users/nefelim584/PycharmProjects/JEV_Judge/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 170/170 [00:00<00:00, 5582.73it/s]


10:49:10 | INFO    | jevlite: load 6.7s, 11.8 rec/s grouped, 0 errors


/Users/nefelim584/PycharmProjects/JEV_Judge/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 394/394 [00:00<00:00, 472.50it/s]


10:58:53 | INFO    | nli: load 2.6s, 0.8 rec/s grouped, 0 errors
11:00:20 | INFO    | laya: load 31.2s, 11.2 rec/s grouped, 0 errors


,predictor,load_s,memory_gb_loaded,memory_gb_after,throughput_rec_s,errors
model,,,,,,
jevlite,jevlite:stage_a-full-s42,6.72,2.01,2.02,11.78,0
nli,nli:MoritzLaurer/DeBERTa-v3-large-mnli-fever-a...,2.61,2.01,5.05,0.78,0
laya,laya:base,31.22,2.01,2.03,11.21,0


## 5. JEV (API)

In [6]:
from jevlite.data.unified import load_predictions

if RUN_JEV_LIVE:
    import sys

    from dotenv import load_dotenv
    from typesafe_sdk import TypeSafeClient

    sys.path.insert(0, str(ROOT / "scripts"))
    import predict_jev_api as jev_api

    load_dotenv()
    live = random.Random(SEED).sample(bench, min(N_JEV_LIVE, len(bench)))
    with TypeSafeClient(api_key=os.environ["TYPESAFE_API_KEY"], model=JEV_MODEL) as client:
        for r in live[:3]:  # warm the connection (TLS, keep-alive)
            client.system_one(state=r.state, questions={"q": jev_api._question(r)})
        for r in live:
            t = time.perf_counter()
            client.system_one(state=r.state, questions={"q": jev_api._question(r)})
            latency_rows.append({"model": "jev", "id": r.id, "latency_ms": (time.perf_counter() - t) * 1000, "error": None})
    jev_source = f"live, {len(live)} calls one at a time"
else:
    recorded = load_predictions(JEV_PREDS)
    latency_rows += [{"model": "jev", "id": r.id, "latency_ms": recorded[r.id].latency_ms, "error": None}
                     for r in bench if r.id in recorded]
    jev_source = f"recorded in {JEV_PREDS} (8 requests in flight)"
infos.append({"model": "jev", "predictor": f"jev API ({jev_source})"})
jev_source

'recorded in data/jev_api/preds.jsonl (8 requests in flight)'

## 6. Results

In [7]:
MODELS = ["nli", "laya", "jevlite", "jev"]  # colour slots as in STAGE_A.md / STAGE_A_API.md
LABELS = {"nli": "NLI (DeBERTa-v3-large)", "laya": "Laya", "jevlite": "jev-lite", "jev": "JEV (API)"}

lat = pd.DataFrame(latency_rows).merge(meta, on="id")
ok = lat[lat["error"].isna()]


def q(p):
    f = lambda s: s.quantile(p)
    f.__name__ = f"p{int(p * 100)}"
    return f


summary = ok.groupby("model")["latency_ms"].agg(["count", "mean", q(0.5), q(0.9), q(0.95)])
by_prim = ok.pivot_table(index="model", columns="primitive", values="latency_ms", aggfunc="median").add_prefix("p50 ")
info_df = pd.DataFrame(infos).set_index("model")
table = summary.join(by_prim).join(info_df[["throughput_rec_s", "load_s", "memory_gb_loaded"]], how="left")
table = table.reindex([m for m in MODELS if m in table.index]).rename(index=LABELS)
table.round(1)

,count,mean,p50,p90,p95,p50 bool,p50 choice,p50 score,throughput_rec_s,load_s,memory_gb_loaded
model,,,,,,,,,,,
NLI (DeBERTa-v3-large),300,641.9,240.6,1895.3,2349.4,121.3,197.0,1085.1,0.8,2.6,2.0
Laya,300,91.2,77.1,160.0,164.9,69.8,52.6,124.8,11.2,31.2,2.0
jev-lite,300,87.5,73.1,152.4,157.2,68.9,54.4,112.4,11.8,6.7,2.0
JEV (API),300,201.6,195.4,224.5,238.8,194.2,196.9,195.3,NaN,NaN,NaN


In [8]:
import json

lat.to_csv(OUT / "latency.csv", index=False)
(OUT / "summary.json").write_text(json.dumps({
    "env": ENV, "n_records": len(bench), "n_per_primitive": N_PER_PRIMITIVE, "jev_source": jev_source,
    "models": infos, "latency_ms": summary.round(2).reset_index().to_dict("records"),
}, indent=1, default=str))
print(f"→ {OUT}/latency.csv, {OUT}/summary.json")

→ data/speed/apple_silicon/latency.csv, data/speed/apple_silicon/summary.json


## 7. Charts

1. Latency per model and primitive: dot = median, line = 10th–90th percentile, log scale.
2. Median latency by input length (characters, quintiles of the subset).
3. Throughput of the local models with grouped calls.

In [9]:
import numpy as np

from matplotlib.ticker import FixedLocator, NullLocator, StrMethodFormatter

from jevlite.reports.style import SURFACE, TEXT_SECONDARY, save_figure, series_colors, setup_matplotlib

plt = setup_matplotlib()
COLORS = series_colors(MODELS)
present = [m for m in MODELS if m in set(ok["model"])]
TICKS = [5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000]


def plain_log_axis(axis, lo, hi):
    # log scale with plain-number ticks instead of 6×10² labels
    axis.set_major_locator(FixedLocator([t for t in TICKS if lo / 1.5 <= t <= hi * 1.5]))
    axis.set_minor_locator(NullLocator())
    axis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))

rows = [(m, p) for m in present for p in ("bool", "choice", "score")]
fig, ax = plt.subplots(figsize=(7.6, 0.32 * len(rows) + 1.6))
for i, (m, p) in enumerate(rows):
    s = ok[(ok["model"] == m) & (ok["primitive"] == p)]["latency_ms"]
    if s.empty:
        continue
    ax.plot([s.quantile(0.1), s.quantile(0.9)], [i, i], color=COLORS[m], linewidth=2, zorder=1)
    ax.scatter([s.median()], [i], s=64, color=COLORS[m], edgecolor=SURFACE, linewidth=2, zorder=2)
    ax.text(s.quantile(0.9) * 1.08, i, f"{s.median():.0f} ms", va="center", fontsize=9, color=TEXT_SECONDARY)
ax.set_yticks(range(len(rows)), [f"{LABELS[m]} · {p}" for m, p in rows])
ax.set_xscale("log")
plain_log_axis(ax.xaxis, ok["latency_ms"].quantile(0.05), ok["latency_ms"].quantile(0.95))
ax.set_ylim(len(rows) - 0.3, -0.7)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
ax.set_xlabel("latency per record, one call at a time (ms, log scale)")
ax.set_title(f"Latency on {ENV['chip']} (MPS) vs JEV over the API", loc="left")
print(save_figure(fig, OUT, "latency_by_primitive")[0])
fig

figures/latency_by_primitive.png


<Figure size 760x544 with 1 Axes>

In [10]:
ok = ok.assign(length=pd.qcut(ok["chars"], 5))
med = ok.groupby(["model", "length"], observed=True)["latency_ms"].median().reset_index()
mid = {iv: (iv.left + iv.right) / 2 for iv in med["length"].unique()}

fig, ax = plt.subplots(figsize=(7.6, 4.2))
for m in present:
    d = med[med["model"] == m].sort_values("length")
    x = [mid[iv] for iv in d["length"]]
    ax.plot(x, d["latency_ms"], color=COLORS[m], marker="o", markersize=8, markeredgecolor=SURFACE, markeredgewidth=2, label=LABELS[m])
ax.set_xscale("log")
ax.set_yscale("log")
plain_log_axis(ax.yaxis, med["latency_ms"].min(), med["latency_ms"].max())
plain_log_axis(ax.xaxis, min(mid.values()), max(mid.values()))
ax.set_xlabel("input length (characters: state + question + options), quintile midpoints")
ax.set_ylabel("median latency (ms)")
ax.set_title("Median latency by input length", loc="left")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=len(present))
print(save_figure(fig, OUT, "latency_vs_length")[0])
fig

figures/latency_vs_length.png


<Figure size 760x420 with 1 Axes>

In [11]:
thr = info_df.dropna(subset=["throughput_rec_s"]).reindex([m for m in MODELS if m in info_df.index]).dropna(subset=["throughput_rec_s"])

fig, ax = plt.subplots(figsize=(7.6, 0.5 * len(thr) + 1.4))
ax.barh(range(len(thr)), thr["throughput_rec_s"], height=0.5, color=[COLORS[m] for m in thr.index], edgecolor=SURFACE, linewidth=2)
for i, v in enumerate(thr["throughput_rec_s"]):
    ax.text(v, i, f"  {v:.1f} rec/s", va="center", fontsize=9, color=TEXT_SECONDARY)
ax.set_yticks(range(len(thr)), [LABELS[m] for m in thr.index])
ax.set_ylim(len(thr) - 0.5, -0.5)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
ax.set_xlabel("records per second, grouped calls (as predict_baseline.py)")
ax.set_title("Throughput on MPS (local models only)", loc="left")
print(save_figure(fig, OUT, "throughput")[0])
fig

figures/throughput.png


<Figure size 760x290 with 1 Axes>

## 8. Reading the numbers

- **Local vs API is not like for like.** JEV's latency includes the network round trip and TypeSafe's server; the local numbers are this Mac's GPU only. For a judge in production both matter: local means no network and no per-call cost, the API means no hardware.
- **NLI scales with the number of options:** it scores one premise–hypothesis pair per option, so Choice with many options is slow; jev-lite and Laya read all options in one pass.
- **JEV throughput** is not measured here: over the API it scales with requests in flight (rate limits permitting), not with this machine.
- **Recorded JEV latencies** come from a run with 8 requests in flight; one at a time (`RUN_JEV_LIVE = True`) can be a little faster.
- **One run, one machine:** repeat the notebook to see the run-to-run spread; first-run numbers after a reboot include cold file caches in `load_s`.